# Geração da Base Simulada (People Analytics)
Este notebook gera uma base de dados simulada, para o projeto ***Turnover Crítico***, que considera o cenário de uma instituição financeira fictícia com aproximadamente 2.500 colaboradores.


A base criada simula de forma intencional, padrões reais observados no mercado, como por exemplo:
- Turnover elevado em Comercial e Tecnologia
- Queda de engajamento ao longo do tempo
- Desigualdade em promoções para análise de equidade
- Custos de reposição altos em níveis estratégicos

Todos os dados seguem princípios de:
- Reprodutibilidade (seed fixa)
- Ética e privacidade (dados 100% sintéticos)
- Estrutura de People Analytics corporativo

## 1. Imports e Configurações (Python)

In [1]:
import time
start = time.perf_counter()

# ----------------------------
# Configuração inicial
# ----------------------------

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Garantir reprodutibilidade da simulação
np.random.seed(42)

# Número de colaboradores simulados
n_colaboradores = 2500

# Identificador único do colaborador
colaborador_id = np.arange(1, n_colaboradores + 1)

## 2. Geração da Tabela de Estrutura Organizacional

**Racional de negócio**
- Considerei utilizar atributos que expliquem promoções, engajamento, turnover. Para isso, tentei estruturar aspectos similares a um RH corporativo real. Também tentei buscar distribuições controladas por área, nível e salários.

In [2]:
# ----------------------------
# Estrutura organizacional
# ----------------------------

# Distribuição de colaboradores por área
areas = [
    "Comercial",
    "Tecnologia",
    "Risco & Compliance",
    "Crédito",
    "Produtos Digitais",
    "Backoffice"
]

prob_area = [0.28, 0.22, 0.12, 0.10, 0.13, 0.15]  # distribuição aproximada do setor

colaborador_area = np.random.choice(
    areas,
    size=n_colaboradores,
    p=prob_area
)

# Níveis de cargo (pirâmide organizacional)
niveis_cargo = ["Júnior", "Pleno", "Sênior", "Líder"]
prob_nivel = [0.35, 0.40, 0.20, 0.05]

colaborador_nivel = np.random.choice(
    niveis_cargo,
    size=n_colaboradores,
    p=prob_nivel
)

In [3]:
# ----------------------------
# Demografia (para análise de equidade)
# ----------------------------

generos = ["F", "M", "Outro"]
prob_genero = [0.47, 0.50, 0.03]

racas = ["Branca", "Parda", "Preta", "Amarela", "Indígena", "Não declarado"]
prob_raca = [0.46, 0.38, 0.09, 0.04, 0.01, 0.02]

colaborador_genero = np.random.choice(
    generos,
    size=n_colaboradores,
    p=prob_genero
)

colaborador_raca = np.random.choice(
    racas,
    size=n_colaboradores,
    p=prob_raca
)

In [4]:
# ----------------------------
# Informações contratuais
# ----------------------------

# Data de admissão simulada entre 2015 e 2025
data_inicio_admissao = datetime(2015, 1, 1)

colaborador_data_admissao = [
    data_inicio_admissao + timedelta(days=int(np.random.uniform(0, 365 * 10)))
    for _ in range(n_colaboradores)
]

# Idade média simulada (com limites realistas)
colaborador_idade = np.clip(
    np.random.normal(loc=34, scale=6, size=n_colaboradores).astype(int),
    21,
    60
)

In [5]:
# ----------------------------
# Estrutura salarial
# ----------------------------

# Salário base por nível de cargo
salario_base_por_nivel = {
    "Júnior": 3000,
    "Pleno": 6000,
    "Sênior": 10000,
    "Líder": 15000
}

# Multiplicador salarial por área (diferença de mercado)
multiplicador_area = {
    "Comercial": 1.00,
    "Tecnologia": 1.25,
    "Produtos Digitais": 1.35,
    "Risco & Compliance": 1.10,
    "Crédito": 1.15,
    "Backoffice": 0.95
}

# Cálculo do salário inicial com pequena variação individual
colaborador_salario_inicial = np.array([
    salario_base_por_nivel[nivel] *
    multiplicador_area[area] *
    np.random.uniform(0.85, 1.15)
    for nivel, area in zip(colaborador_nivel, colaborador_area)
])

In [6]:
# ----------------------------
# Construção da dimensão de colaboradores
# ----------------------------

dim_colaboradores = pd.DataFrame({
    "colaborador_id": colaborador_id,
    "area": colaborador_area,
    "nivel_cargo": colaborador_nivel,
    "genero": colaborador_genero,
    "raca": colaborador_raca,
    "data_admissao": colaborador_data_admissao,
    "idade": colaborador_idade,
    "salario_base_inicial": colaborador_salario_inicial
})

dim_colaboradores.head()

,colaborador_id,area,nivel_cargo,genero,raca,data_admissao,idade,salario_base_inicial
0,1,Tecnologia,Sênior,F,Parda,2018-09-25,37,11366.628604
1,2,Backoffice,Sênior,M,Parda,2018-04-30,27,10410.136524
2,3,Produtos Digitais,Pleno,M,Branca,2016-10-04,34,9098.936870
3,4,Risco & Compliance,Líder,F,Branca,2021-01-25,33,18049.780472
4,5,Comercial,Júnior,M,Parda,2019-10-06,35,2824.598271


## 3. Geração da um Histórico Mensal de RH

**Racional de negócio**
- Peguei uma janela de 24 meses e apliquei sobre essa janela fatores que apontam qu engajamento cai antes do turnover, performance contribuem promoções e horas extras são fatores que influenciam burnout.

In [7]:
# ----------------------------
# Histórico mensal de RH
# ----------------------------

# Período de análise (24 meses)
datas = pd.date_range(start="2024-01-01", end="2025-12-01", freq="MS")

registros = []

for data in datas:
    
    for i in range(n_colaboradores):
        
        area = dim_colaboradores.loc[i, "area"]
        
        # ----------------------------
        # Engajamento
        # ----------------------------
        # Base média de engajamento organizacional
        
        base_engajamento = np.random.normal(75, 10)
        
        # Queda progressiva em áreas com maior pressão
        if area in ["Comercial", "Tecnologia"]:
            
            meses_passados = (data.year - 2024) * 12 + data.month
            
            base_engajamento -= meses_passados * 0.7
        
        engajamento = np.clip(base_engajamento, 0, 100)
        
        
        # ----------------------------
        # Performance
        # ----------------------------
        
        nivel = dim_colaboradores.loc[i, "nivel_cargo"]
        
        ajuste_performance = {
            "Júnior": -0.2,
            "Pleno": 0,
            "Sênior": 0.2,
            "Líder": 0.3
        }
        
        performance = np.clip(
            np.random.normal(3 + ajuste_performance[nivel], 0.6),
            1,
            5
        )
        
        
        # ----------------------------
        # Carga de trabalho (horas extras)
        # ----------------------------
        
        horas_extras = max(0, np.random.normal(12, 8))
        
        if area == "Tecnologia":
            horas_extras += np.random.uniform(4, 12)
        
        
        registros.append([
            dim_colaboradores.loc[i, "colaborador_id"],
            data,
            engajamento,
            performance,
            horas_extras
        ])


# Construção da tabela mensal de RH

fato_rh_mensal = pd.DataFrame(
    registros,
    columns=[
        "colaborador_id",
        "mes_ano",
        "engajamento",
        "performance",
        "horas_extras"
    ]
)

fato_rh_mensal.head()

,colaborador_id,mes_ano,engajamento,performance,horas_extras
0,1,2024-01-01,66.406476,2.336944,12.857885
1,2,2024-01-01,86.961337,2.795820,4.655934
2,3,2024-01-01,68.444575,3.299464,10.177830
3,4,2024-01-01,82.142375,2.626891,17.575977
4,5,2024-01-01,78.637934,1.895322,24.049866


## 4. Promoções (com viés sutil para teste de equidade)

- Adotei uma menor probabilidade de promoção para mulheres em Tecnologia (≈ −15%)

In [8]:
# ----------------------------
# Simulação de promoções
# ----------------------------

registros_promocoes = []

for i in range(n_colaboradores):
    
    genero = dim_colaboradores.loc[i, "genero"]
    area = dim_colaboradores.loc[i, "area"]
    nivel = dim_colaboradores.loc[i, "nivel_cargo"]
    
    
    # Probabilidade base de promoção por nível
    
    prob_promocao = {
        "Júnior": 0.20,
        "Pleno": 0.15,
        "Sênior": 0.10,
        "Líder": 0.02
    }[nivel]
    
    
    # Pequeno viés estrutural simulado para análise de equidade
    
    if area == "Tecnologia" and genero == "F":
        prob_promocao *= 0.85
    
    
    # Simulação da promoção
    
    if np.random.rand() < prob_promocao:
        
        nivel_novo = {
            "Júnior": "Pleno",
            "Pleno": "Sênior",
            "Sênior": "Líder",
            "Líder": "Líder"
        }[nivel]
        
        registros_promocoes.append([
            dim_colaboradores.loc[i, "colaborador_id"],
            np.random.choice(datas),
            nivel,
            nivel_novo
        ])


eventos_promocoes = pd.DataFrame(
    registros_promocoes,
    columns=[
        "colaborador_id",
        "data_promocao",
        "nivel_anterior",
        "nivel_novo"
    ]
)

eventos_promocoes.head()

,colaborador_id,data_promocao,nivel_anterior,nivel_novo
0,1,2024-05-01,Sênior,Líder
1,8,2024-11-01,Pleno,Sênior
2,13,2024-07-01,Pleno,Sênior
3,15,2025-01-01,Pleno,Sênior
4,16,2024-12-01,Sênior,Líder


## 5. Desligamentos (case de turnover problemático)
 
- Inseri um cenário de maior probabilidade em Comercial e TI, com fatores que determinam que engajamento baixo aumenta risco de desligamento voluntário e que os custos de reposição dependem do cargo.

In [9]:
# ----------------------------
# Simulação de desligamentos
# ----------------------------

registros_desligamentos = []

for i in range(n_colaboradores):
    
    area = dim_colaboradores.loc[i, "area"]
    nivel = dim_colaboradores.loc[i, "nivel_cargo"]
    
    
    # Probabilidade base de saída
    
    prob_saida = 0.08 if area in ["Comercial", "Tecnologia"] else 0.04
    
    
    # Ajuste por engajamento médio
    
    engajamento_medio = fato_rh_mensal[
        fato_rh_mensal.colaborador_id == i + 1
    ]["engajamento"].mean()
    
    
    if engajamento_medio < 60:
        prob_saida *= 1.5
    
    
    # Simulação de saída
    
    if np.random.rand() < prob_saida:
        
        data_saida = np.random.choice(datas)
        
        tipo_saida = np.random.choice(
            ["Voluntário", "Involuntário"],
            p=[0.7, 0.3]
        )
        
        
        custo_reposicao = (
            salario_base_por_nivel[nivel]
            * multiplicador_area[area]
            * np.random.uniform(2.5, 4.5)
        )
        
        registros_desligamentos.append([
            dim_colaboradores.loc[i, "colaborador_id"],
            data_saida,
            tipo_saida,
            custo_reposicao
        ])


eventos_desligamentos = pd.DataFrame(
    registros_desligamentos,
    columns=[
        "colaborador_id",
        "data_desligamento",
        "tipo_desligamento",
        "custo_reposicao"
    ]
)

eventos_desligamentos.head()

,colaborador_id,data_desligamento,tipo_desligamento,custo_reposicao
0,42,2024-02-01,Involuntário,35928.607687
1,59,2024-12-01,Voluntário,10358.210415
2,128,2024-04-01,Voluntário,19796.181817
3,131,2024-10-01,Voluntário,19138.336736
4,169,2025-11-01,Voluntário,18134.836231


## 6. Exportar arquivos

In [10]:
# ----------------------------
# Exportação dos dados simulados
# ----------------------------

dim_colaboradores.to_csv(
    "../data/raw/dim_colaboradores.csv",
    index=False
)

fato_rh_mensal.to_csv(
    "../data/raw/fato_rh_mensal.csv",
    index=False
)

eventos_promocoes.to_csv(
    "../data/raw/eventos_promocoes.csv",
    index=False
)

eventos_desligamentos.to_csv(
    "../data/raw/eventos_desligamentos.csv",
    index=False
)

print("Arquivos gerados e gravados com sucesso!")

Arquivos gerados e gravados com sucesso!


In [11]:
end = time.perf_counter()
print(f"Tempo total: {end - start:.2f} segundos")

Tempo total: 25.51 segundos
